# Zaskaleta AI Twin — Free Colab GPU
Український текст → MMS Ukrainian TTS → OpenVoice V2 → MuseTalk 1.5 → MP4 9:16.

Запускайте клітинки зверху вниз. Master-фото і Master-голос не завантажуються в GitHub.

In [ ]:
import torch, subprocess, os
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU не активний. Runtime → Change runtime type → T4 GPU')
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
!rm -rf /content/zaskaleta-ai-twin-colab
!git clone --depth 1 https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git /content/zaskaleta-ai-twin-colab
WORKER='/content/zaskaleta-ai-twin-colab/worker'
MUSETALK='/content/MuseTalk'
print('✅ Public AI Twin code loaded')


In [ ]:
env=os.environ.copy()
env['APP_DIR']=WORKER
env['MUSETALK_ROOT']=MUSETALK
subprocess.run(['bash', f'{WORKER}/install_gpu_engines.sh'], env=env, check=True)
print('✅ OpenVoice + MuseTalk installed')


## 1. Завантажте MASTER PHOTO

In [ ]:
from google.colab import files
from pathlib import Path
up=files.upload()
PHOTO='/content/' + next(iter(up.keys()))
print('PHOTO =', PHOTO)


## 2. Завантажте MASTER VOICE — WAV або MP3

In [ ]:
up=files.upload()
VOICE='/content/' + next(iter(up.keys()))
print('VOICE =', VOICE)


## 3. Введіть український текст

In [ ]:
TEXT = 'Привіт. Це тест мого AI-двійника. Я говорю українською своїм голосом.' #@param {type:"string"}
SCRIPT='/content/zaskaleta_script.txt'
Path(SCRIPT).write_text(TEXT, encoding='utf-8')
print(TEXT)


In [ ]:
VOICE_OUT='/content/zaskaleta_voice.wav'
subprocess.run(['python3', f'{WORKER}/voice_mms_openvoice.py', '--script', SCRIPT, '--voice', VOICE, '--output', VOICE_OUT, '--language', 'uk'], check=True)
print('✅ Voice ready:', VOICE_OUT)


In [ ]:
RAW_VIDEO='/content/zaskaleta_raw.mp4'
env=os.environ.copy(); env['MUSETALK_ROOT']=MUSETALK
subprocess.run(['python3', f'{WORKER}/lipsync_musetalk.py', '--photo', PHOTO, '--audio', VOICE_OUT, '--output', RAW_VIDEO], env=env, check=True)
print('✅ Lip-sync ready:', RAW_VIDEO)


In [ ]:
FINAL='/content/Zaskaleta_AI_Twin_9x16.mp4'
vf='scale=1080:1920:force_original_aspect_ratio=decrease,pad=1080:1920:(ow-iw)/2:(oh-ih)/2,setsar=1'
subprocess.run(['ffmpeg','-y','-i',RAW_VIDEO,'-vf',vf,'-c:v','libx264','-preset','medium','-crf','19','-c:a','aac','-b:a','192k','-movflags','+faststart',FINAL], check=True)
print('✅ FINAL:', FINAL)


In [ ]:
from IPython.display import Video, display
display(Video(FINAL, embed=True, width=360))
files.download(FINAL)
